# Phase 4 - Chemprop (D-MPNN), the second external baseline

**Run this in a fresh Colab runtime and nothing else.** `pip install chemprop` pins its own
torch and lightning and will move the versions every other result in this project was
produced under. That is not a theoretical worry here: this project measured that the *same*
code and seed on a different accelerator is already enough to move 18% of single-split
numbers by more than the minimum detectable effect. A dependency rewrite is a bigger
perturbation than that.

So Chemprop gets its own runtime, its own bundle, and its own Drive folder, and the runner
was made **torch-free** on purpose -- it reads split indices from JSON and computes metrics
with numpy and scikit-learn, so a rewritten torch cannot break it.

**What is held identical.** The molecules and the split assignment. Chemprop 2.x has no
`--separate-val-path`; splits are passed as a `split` column in one CSV holding
`train`/`val`/`test` per row, built from this project's split indices. Chemprop therefore
never sees its own scaffold splitter and never touches test during training.

**What is not, unavoidably.** Featurisation, optimiser schedule, early stopping and internal
target scaling are Chemprop's own. That is the honest form of an external-baseline comparison
and the paper says so. Metrics are recomputed from its saved predictions with this project's
`metrics.py`, so AUC and RMSE mean the same thing as every other row -- NaN-masked, per task,
in chemical units. Chemprop reports regression error on its internally scaled target; taking
that at face value would repeat a bug this project already fixed once.

Expected: **~4-6 h on a T4** for 8 datasets x 6 splits. There is no `--resume`, so if the
session drops, narrow `--variants` to what is left and re-run.

## 1. Check you actually got a GPU

In [ ]:
import torch
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))
else: print('No GPU. Runtime > Change runtime type > T4 GPU, then re-run.')

## 2. Connect your Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**Build the bundle first** — it is a build product and is not stored in the repository:

```
python -m scripts.make_colab_bundle --preset fusion --out mpp_chemprop_bundle.zip
```

## 3. Point at the bundle

Upload `mpp_chemprop_bundle.zip` (4 MB - Chemprop needs only SMILES, labels and the split
indices) to Drive first.

In [ ]:
BUNDLE = '/content/drive/MyDrive/mpp_chemprop_bundle.zip'  # edit if elsewhere
OUTDIR = '/content/drive/MyDrive/mpp_chemprop'

import os
assert os.path.exists(BUNDLE), f'Not found: {BUNDLE} -- check path, re-run.'
os.makedirs(OUTDIR, exist_ok=True)
print('bundle:', round(os.path.getsize(BUNDLE)/1e6, 1), 'MB')

## 4. Unpack, then install Chemprop

The install may downgrade torch and will print dependency warnings about the runtime's
preinstalled packages. That is expected and is the reason this notebook is separate. **Restart
the runtime if Colab asks**, then re-run cells 2-4 (not cell 5 onward) before continuing.

In [ ]:
import zipfile, os
WORK = '/content/mpp'
os.makedirs(WORK, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z: z.extractall(WORK)
os.chdir(WORK)

!pip -q install chemprop

import shutil, subprocess
exe = shutil.which('chemprop')
print('chemprop on PATH:', exe)
assert exe, 'chemprop did not install -- read the pip output above.'
print(subprocess.run(['chemprop','--help'], capture_output=True, text=True).stdout[:400])

## 5. Keep results on Drive

In [ ]:
import os
os.makedirs(f'{OUTDIR}/runs', exist_ok=True)
os.makedirs('results', exist_ok=True)
if not os.path.islink('results/runs'):
    if os.path.exists('results/runs'):
        import shutil; shutil.rmtree('results/runs')
    os.symlink(f'{OUTDIR}/runs', 'results/runs')
print('results/runs ->', os.path.realpath('results/runs'))

## 6. Smoke test - do not skip this

This already earned its place once: the first attempt died because Chemprop refuses to start
unless `epochs > warmup_epochs`, and its warmup default is 2. Invisible at 50 epochs, fatal at
2. Fixed, but the lesson stands.

It now covers **both risky paths**, not just the easy one:

* **freesolv** - regression, one task, 642 molecules. Fast.
* **tox21** - classification, **twelve** tasks, and **missing labels**. This is the path that
  can go wrong silently: twelve prediction columns read back in the wrong order would scramble
  twelve tasks against their labels and still produce plausible-looking numbers.

Both are checked for row alignment against the split index, and tox21 is additionally checked
for column count. A few minutes now against a five-hour run that produces quiet nonsense.

In [ ]:
import numpy as np, json, os

def smoke(ds, epochs=3):
    rc = os.system(f'python -m src.baselines.chemprop_runner --datasets {ds} '
                   f'--variants deepchem --epochs {epochs} --accelerator gpu --tag _smoke')
    if rc != 0:
        print(f'{ds}: RUN FAILED (exit {rc}) - read the traceback above, do not continue')
        return False
    p = np.load(f'results/preds/{ds}__smoke_test.npy')
    idx = json.load(open(f'data/splits/{ds}_deepchem.json'))['test']
    y = np.load(f'data/pool/{ds}_ecfp.npz', allow_pickle=True)['y']
    ok_rows, ok_cols = p.shape[0] == len(idx), p.shape[1] == y.shape[1]
    print(f'{ds}: preds {p.shape} | expected ({len(idx)}, {y.shape[1]}) | '
          f'rows {"OK" if ok_rows else "MISALIGNED"} | cols {"OK" if ok_cols else "WRONG"}')
    return ok_rows and ok_cols

good = smoke('freesolv') & smoke('tox21')

!rm -f results/metrics/*_smoke*.csv results/preds/*_smoke*.npy
!rm -rf results/runs/deepchem/metrics/*_smoke* results/runs/deepchem/preds/*_smoke*
print()
print('SMOKE TEST PASSED - run cell 7' if good else 'STOP. Fix before running cell 7.')

## 7. Train

No `--resume`: Chemprop owns its own training loop and this runner does not archive
mid-variant. If the session drops, look at cell 8 and narrow `--variants` to the ones still
missing.

In [ ]:
!python -m src.baselines.chemprop_runner     --variants deepchem seed0 seed1 seed2 seed3 seed4     --epochs 50 --accelerator gpu

## 8. Check what finished

In [ ]:
import glob
allok = True
for v in ['deepchem','seed0','seed1','seed2','seed3','seed4']:
    m = len(glob.glob(f'{OUTDIR}/runs/{v}/metrics/*_chemprop_*.csv'))
    p = len(glob.glob(f'{OUTDIR}/runs/{v}/preds/*_chemprop_*.npy'))
    done = m == 16 and p == 16
    allok &= done
    print(v, 'metrics', m, 'preds', p, 'ok' if done else 'INCOMPLETE')
print()
print('ALL DONE - run cell 9' if allok else
      'Re-run cell 7 with --variants limited to the incomplete ones above.')

## 9. Bring the results home

Then, locally, **with `--dry-run` first**:

```
python -m scripts.merge_colab_results ~/Downloads/chemprop_results.zip --dry-run
```

`chemprop` is a new tag, so this merge adds rows and collides with nothing.

In [ ]:
import shutil, os
out = shutil.make_archive('/content/chemprop_results', 'zip', f'{OUTDIR}/runs')
print('wrote', out, round(os.path.getsize(out)/1e6, 2), 'MB')
from google.colab import files
files.download(out)